In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 잘 지냈어요! 당신은 어떻게 지내고 계신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 12, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb86cfd659', 'id': 'chatcmpl-EjGiA8Egv6PKTibfgxAPitWny65XO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ce03-9c63-7038-8586-8eb85039fd8a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 18, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [3]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9123aa2e8c', 'id': 'chatcmpl-EjGipZ55rjFM3QF8kOCDDyfHgg8HT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1ce04-389e-7667-bcc5-f9b31adf1770-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'A

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-11-12 13:29:07 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9123aa2e8c', 'id': 'chatcmpl-EjGipZ55rjFM3QF8kOCDDyfHgg8HT', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1ce04-389e-7667-bcc5-f9b31adf1770-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='현재 부산은 2026년 11월 12일 오후 1시 29분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 192, 'total_tokens': 215, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9123aa2e8c', 'id': 'chatcmpl-EjGisYGb1XAULHodpBelVMSaiJ6aU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ce04-488c-715d-a1ae-3eb8ece21e7d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 23, 'total_tokens': 215, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")


In [8]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [9]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 283, 'total_tokens': 310, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fc3bf93e48', 'id': 'chatcmpl-EjGixhsniMS667GHWYNJQZTSVPU2P', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a1ce04-58fe-77ac-8a0f-508c6ef481b4-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_BK7Z8R1G8UkcK6PfeubleIYv', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 283, 'output_tokens': 27, 'total_tokens

In [10]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-10-12 00:00:00-04:00 | 382.61 | 382.7  | 377.58 |  378.03 | 2.02443e+07 |           0 |              0 |\n| 2026-10-13 00:00:00-04:00 | 372.33 | 377.05 | 371.3  |  375.23 | 1.8393e+07  |           0 |              0 |\n| 2026-10-14 00:00:00-04:00 | 371.16 | 371.39 | 363.82 |  367.16 | 3.10014e+07 |           0 |              0 |\n| 2026-10-15 00:00:00-04:00 | 360.57 | 363.75 | 355.23 |  362.44 | 2.67283e+07 |           0 |              0 |\n| 2026-10-16 00:00:00-04:00 | 369.15 | 370.5  | 349.33 |  354.72 | 6.20673e+07 |           0 |              0 |\n| 2026-10-19 00:00:00-04:00 | 357.5  | 369.58 | 354.47 |  368.06 | 3.88179e+07 |           0 |              0 |\n| 2026-10-20 00:00:00-04:0

In [11]:
llm_with_tools.invoke(messages)

AIMessage(content='한 달 전인 2026년 10월 12일의 테슬라 주가는 **$378.03**로 마감했습니다. 현재 주가(2026년 11월 11일)는 **$390.84**입니다. \n\n따라서, 테슬라의 주가는 한 달 전보다 **올랐습니다.**', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 1703, 'total_tokens': 1779, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fc3bf93e48', 'id': 'chatcmpl-EjGiztbgAuoDb0xTupYy0x7UTSn68', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ce04-6528-71f0-ada4-c9e4679d3f84-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1703, 'output_tokens': 76, 'total_tokens': 1779, 'input_token_de

## 08-4 스트림 출력

In [12]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end='|') 

|안|녕하세요|!| 한국| 사회|의| 문제|점|에| 대해| 이야기|해|보|겠습니다|.| 

|1|.| **|청|년|실|업|**|:| 경제|가| 성장|하고| 있음|에도| 불|구|하고| 많은| 청|년|들이| 취|업|에| 어려|움을| 겪|고| 있습니다|.| 경쟁|이| 치|열|하고|,| 대|기업| 중심|의| 채|용|이| 문제|로| 지|적|되고| 있습니다|.

|2|.| **|과|도|한| 학|벌| 중심| 사회|**|:| 학|벌|과| 스|펙|이| 중요한| 사회|적| 기준|으로| 자리| 잡|으|면서|,| 학생|들이| 과|도|한| 학|습| 스트|레|스를| 받|는| 경우|가| 많|습니다|.| 이는| 교육|의| 질| 저|하|와| 함께| 학|습|의| 본|래| 목적|을| 훼|손|할| 수| 있습니다|.

|3|.| **|성|차|별|과| 성|폭|력|**|:| 여|전히| 성|별|에| 따른| 차|별|과| 성|폭|력| 문제가| 심|각|합니다|.| 특히| 직|장에서|의| 성|희|롱| 사건|이나| 여성|의| 경|력| 단|절| 등의| 문제|는| 해결|이| 필요|합니다|.

|4|.| **|고|령|화| 사회|**|:| 인|구| 고|령|화|가| 진행|되|면서| 노|인| 복|지|,| 연|금| 시스템|,| 요|양| 서비스| 등의| 문제가| 대|두|되고| 있습니다|.| 미래| 세|대|의| 부담|도| 우|려|되고| 있습니다|.

|5|.| **|정|신| 건강| 문제|**|:| 우|울|증|,| 불|안|장|애| 등| 정신| 건강| 문제|에| 대한| 사회|적| 인|식|이| 부족|하고|,| 치료|받|기| 어려|운| 상황|이| 많은| 것이| 문제|입니다|.

|6|.| **|부|동|산| 문제|**|:| 급|격|한| 부|동|산| 가격| 상승|으로| 인해| 많은| 사람들이| 주|거| 문제|로| 어려|움을| 겪|고| 있습니다|.| 이는| 젊|은| 세|대|의| 주|거| 안정|성을| 위|협|하고| 있습니다|.

|이| 외|에도| 여러| 복|합|적인| 문제|들이| 존재|하지만|,| 이러한| 문제|들은| 개선|할| 수| 있는|

In [13]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
gathered = None
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk #type:ignore
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)#type:ignore

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_Lg2cQuTkeZTyTIWIFsnhzjVN', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_Lg2cQuTkeZTyTIWIFsnhzjVN', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_Lg2cQuTkeZTyTIWIFsnhzjVN', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_Lg2cQuTkeZTyTIWIFsnhzjVN', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_Lg2cQuTkeZTyTIWIFsnhzjVN', 'type': 'tool_c

In [14]:
gathered

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fc3bf93e48', 'service_tier': 'default'}, id='lc_run--01a1ce0a-59fe-74b9-a214-931f85dec978', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_Lg2cQuTkeZTyTIWIFsnhzjVN', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 23, 'total_tokens': 226, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_Lg2cQuTkeZTyTIWIFsnhzjVN', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

In [15]:
for tool_call in gathered.tool_calls: #type:ignore
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-11-12 13:37:34 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fc3bf93e48', 'service_tier': 'default'}, id='lc_run--01a1ce0a-59fe-74b9-a214-931f85dec978', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_Lg2cQuTkeZTyTIWIFsnhzjVN', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 23, 'total_tokens': 226, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_Lg2cQuTkeZTyTIWIFsnhzjVN', 'index': 0, 'ty

In [16]:
for c in llm_with_tools.stream(messages):
    print(c.content, end='|')

|부|산|은| 지금| |202|6|년| |11|월| |12|일| |13|시| |37|분| |34|초|입니다|.||||